<a href="https://colab.research.google.com/github/Bee-Ene/shri-qlearning-execute-or-clarify/blob/main/QL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q-Learning Main Training

In [ ]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import random
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import csv
from dataclasses import dataclass
from enum import Enum
from typing import Optional, Tuple
from datetime import datetime
from itertools import product as iterproduct

SEED = 42

In [ ]:
# STATE SPACE DEFINITION
# (3 × 2 × 3 = 18 states, all reachable)

class Completeness(Enum):
    """
    How complete is the command?
    Derived from the language understanding module by argument presence (STT + NLU parser).
    """
    COMPLETE   = 0   # All required args present
    PARTIAL    = 1   # Some required args present, some absent
    INCOMPLETE = 2   # All required args absent

class Clarity(Enum):
    """
    How many valid interpretations does the command have?
    Derived from semantic parsing confidence.
    """
    CLEAR     = 0   # parse_confidence >= threshold
    AMBIGUOUS = 1   # parse_confidence <  threshold

class Feasibility(Enum):
    """
    Can the robot physically execute right now?
    Derived from onboard sensors (LiDAR, camera, odometry).
    """
    FEASIBLE  = 0   # Path clear, target confirmed, safe to act
    UNCERTAIN = 1   # Partial occlusion or target not yet confirmed
    BLOCKED   = 2   # Obstacle in path or target not found

UNAMBIGUOUS_INTENTS = {"STOP", "SIT", "STAND"}

# Fixed thresholds (check: sensitivity analysis in run_sensitivity())
PARSE_AMBIGUOUS_THRESHOLD = 0.60
PERC_UNCERTAIN_THRESHOLD  = 0.45
PERC_UNCERTAIN_NAV        = 0.50

def encode_state(c: Completeness, cl: Clarity, f: Feasibility) -> int:
    return c.value * 6 + cl.value * 3 + f.value

def decode_state(s: int) -> Tuple[Completeness, Clarity, Feasibility]:
    c  = Completeness(s // 6)
    r  = s % 6
    cl = Clarity(r // 3)
    f  = Feasibility(r % 3)
    return c, cl, f

In [ ]:
# All 18 state IDs
NUM_STATES  = 18
NUM_ACTIONS = 2
ALL_STATES  = list(range(NUM_STATES))


def state_uncertainty_level(s: int) -> int:
    """
    Scalar uncertainty score: 0 (best) to 5 (worst).
    INCOMPLETE(+2) + AMBIGUOUS(+1) + BLOCKED(+2) = 5 maximum.
    Used to measure whether clarification improved the state.
    Never used for reward labelling.
    """
    c, cl, f = decode_state(s)
    score = 0
    if c == Completeness.INCOMPLETE: score += 2
    elif c == Completeness.PARTIAL:  score += 1
    if cl == Clarity.AMBIGUOUS:      score += 1
    if f == Feasibility.BLOCKED:     score += 2
    elif f == Feasibility.UNCERTAIN: score += 1
    return score

def oracle_should_clarify(c: Completeness, cl: Clarity, f: Feasibility) -> bool:
    """
    Ground-truth oracle only for policy accuracy reporting.

    EXECUTE is correct only for COMPLETE + CLEAR + FEASIBLE (state 0).
    All other states requires CLARIFY.

    Rationale for COMPLETE+CLEAR+UNCERTAIN (state 1):
      P(success) = 0.712.
      Expected reward from direct execution:
        0.712 * 10 + 0.288 * (-10) = +4.24
      Expected reward from clarifying (transitions to FEASIBLE with 75% prob):
        +2 (productive) + 0.9 * (0.95*10 - 0.05*10) = +2 + 8.1 = +10.1
      Clarification is the rational choice (oracle reflects this)
      UNCERTAIN means the robot cannot confirm execution will succeed.
      One targeted clarification resolves this efficiently.
    """
    return not (c == Completeness.COMPLETE and
                cl == Clarity.CLEAR and
                f == Feasibility.FEASIBLE)

In [ ]:
# LOCOMOTION COMMAND DOMAIN
# Defines the vocabulary of commands the Go2 receives.
# In ROS2: this comes from the STT subscriber + NLU parser.

# Locomotion intents the Go2 can execute
INTENTS = ["MOVE", "TURN", "NAVIGATE_TO", "STOP", "SIT", "STAND"]

# Required parameters per intent that the parser must extract
REQUIRED_PARAMS = {
    "MOVE":        ["direction"], # forward/back/left/right
    "TURN":        ["direction"], # left/right/around
    "NAVIGATE_TO": ["target"], # object/location name
    "STOP":        [],
    "SIT":         [],
    "STAND":       [],
}

DIRECTIONS = ["forward", "backward", "left", "right"]
TARGETS    = ["the chair", "the door", "the table", "the person", "the marker"]

@dataclass
class LocoCommand:
    """
    A structured locomotion command as it would appear in the ROS2 system.
    In deployment: produced by the NLU parser from STT text.
    In simulation: generated by simulate_command_parser().
    """
    intent:          str
    direction:       Optional[str]   # for MOVE / TURN
    target:          Optional[str]   # for NAVIGATE_TO
    distance:        Optional[float] # meters, if specified
    raw_text:        str # original utterance
    parse_confidence: float # parser's self-reported confidence [0,1]
    present_params:   list # list of required params that are present
    missing_params:  list  # list of required params that are absent

@dataclass
class PerceptionState:
    """
    Robot's perceptual state from onboard sensors.
    In deployment: produced by LiDAR + camera processing nodes.
    In simulation: generated by simulate_perception().
    """
    path_clear:        bool # LiDAR: no obstacle in direction of travel
    target_visible:    bool # Camera: intended target detected
    num_target_matches: int # how many objects match the description
    obstacle_distance: float # meters to nearest obstacle (inf if none)
    perception_confidence: float # overall sensor confidence [0,1]

In [ ]:
# MULTIMODAL PERCEPTION AND INTERPRETATION LAYER
# Simulates STT + NLU parser + robot sensors

def simulate_command_parser(scenario: dict, noise: float) -> LocoCommand:
    """
    Simulates NLU parser output with realistic noise.
    STOP/SIT/STAND always receive high confidence (no ambiguity possible).
    Completeness tracking: present_params and missing_params are independent lists.

    Takes the ground-truth scenario and returns a structured command as the parser would produce it, with realistic parse errors.

    IN ROS2 DEPLOYMENT: Replace with a subscriber to /command_parsed topic.
    The NLU node (using Whisper for STT + a local LLM for parsing) publishes a LocoCommand-equivalent message. This function's output format is the contract between Layer 1 and Layer 2.
    """
    intent   = scenario["intent"]
    required = REQUIRED_PARAMS[intent]
    direction = scenario.get("direction")
    target    = scenario.get("target")
    distance  = scenario.get("distance")

    # Simulate parse present and missing parameters
    present_params = []
    missing_params = []

    if "direction" in required:
        if direction and random.random() < noise:
            direction = None
        if direction is not None:
            present_params.append("direction")
        else:
            missing_params.append("direction")

    if "target" in required:
        if target and random.random() < noise:
            target = None
        if target is not None:
            present_params.append("target")
        else:
            missing_params.append("target")

    if intent in UNAMBIGUOUS_INTENTS:
        parse_conf = float(np.clip(0.92 + random.gauss(0, 0.02), 0.85, 0.99))
    else:
        n_miss     = len(missing_params)
        # Parse confidence: inversely related to noise and missing params
        base_conf  = 0.90 - (0.28 * noise) - (0.10 * n_miss)
        parse_conf = float(np.clip(base_conf + random.gauss(0, 0.04), 0.05, 0.99))

    # Build raw text (what the human actually said)
    parts = [intent.lower().replace("_", " ")]
    if direction:
      parts.append(direction)
    if target:
      parts.append(f"to {target}")
    if distance:
      parts.append(f"{distance:.1f} meters")

    return LocoCommand(
        intent = intent,
        direction = direction,
        target = target,
        distance = distance,
        raw_text =" ".join(parts),
        parse_confidence = parse_conf,
        present_params = present_params,
        missing_params = missing_params
    )

def simulate_perception(scenario: dict, noise: float) -> PerceptionState:
    """
    Simulates the robot's onboard sensor suite (LiDAR + camera) with realistic noise.

    IN ROS2 DEPLOYMENT: Replace with subscribers to:
      /scan (LiDAR): path_clear, obstacle_distance
      /detected_objects (YOLOv8 on camera): target_visible, num_target_matches
    Confidence is fused from sensor covariance estimates.

    noise: float [0,1], models sensor noise, occlusion, lighting conditions
    """
    num_obstacles = scenario.get("num_obstacles", 0)
    num_targets   = scenario.get("num_targets", 1)

    # Simulate sensor noise on obstacle detection
    detected_obs = num_obstacles
    if random.random() < noise * 0.3:
        detected_obs = max(0, detected_obs + random.choice([-1, 1]))

    path_clear = detected_obs == 0
    obs_dist   = float(np.clip(random.gauss(0.8, 0.2), 0.3, 1.5)) if not path_clear else float('inf')

    # Simulate target detection noise
    detected_tgt = num_targets
    if random.random() < noise * 0.35:
        detected_tgt = max(0, detected_tgt + random.choice([-1, 1]))

    target_visible = detected_tgt > 0

    # Perception confidence: best with clear path + single target
    if path_clear and detected_tgt == 1:
        base_conf = 0.88
    elif not path_clear or detected_tgt == 0:
        base_conf = 0.28
    else:
        base_conf = 0.52

    perc_conf = float(np.clip(base_conf + random.gauss(0, 0.06), 0.05, 0.98))

    return PerceptionState(
        path_clear = path_clear,
        target_visible = target_visible,
        num_target_matches = detected_tgt,
        obstacle_distance = obs_dist,
        perception_confidence = perc_conf
    )

In [ ]:
# STATE ABSTRACTION LAYER
# Converts structured command + perception into discrete QL state.

def build_state(cmd: LocoCommand, perc: PerceptionState) -> int:
    """
    Rule-based fusion: See run_sensitivity() for threshold

    Converts LLM + sensor outputs to a discrete state.
    Completeness: purely from argument presence (independent of confidence).
    Clarity: purely from parse confidence (independent of arg presence).
    Feasibility: purely from sensor data.
    """
    required = REQUIRED_PARAMS[cmd.intent]

    #Completeness
    if not required:
        # Intents with no required params (STOP, SIT, STAND) are always complete
        completeness = Completeness.COMPLETE
    else:
        n_present = len(cmd.present_params)
        n_required = len(required)
        if n_present == n_required:
            completeness = Completeness.COMPLETE
        elif n_present == 0:
            completeness = Completeness.INCOMPLETE
        else:
            completeness = Completeness.PARTIAL

    # Clarity
    if cmd.intent in UNAMBIGUOUS_INTENTS:
        clarity = Clarity.CLEAR
    else:
        clarity = (Clarity.AMBIGUOUS
                   if cmd.parse_confidence < PARSE_AMBIGUOUS_THRESHOLD
                   else Clarity.CLEAR)

    # Feasibility
    needs_target = cmd.intent == "NAVIGATE_TO"
    if not perc.path_clear:
        feasibility = Feasibility.BLOCKED
    elif needs_target:
        if not perc.target_visible:
            feasibility = Feasibility.BLOCKED
        elif perc.num_target_matches > 1:
            feasibility = Feasibility.UNCERTAIN # multiple targets; which one?
        elif perc.perception_confidence < PERC_UNCERTAIN_NAV:
            feasibility = Feasibility.UNCERTAIN
        else:
            feasibility = Feasibility.FEASIBLE
    else:
        # Non-navigation command: feasibility = path clearance
        feasibility = (Feasibility.UNCERTAIN
                       if perc.perception_confidence < PERC_UNCERTAIN_THRESHOLD
                       else Feasibility.FEASIBLE)

    return encode_state(completeness, clarity, feasibility)

In [ ]:
#SENSITIVITY ANALYSIS
# Replaces ablation due to bimodal confidence distribution of the simulator.
# Shows results are stable across threshold ranges
# Justifies fixed threshold choice by robustness rather than grid search

def run_sensitivity_analysis(n_samples: int = 3000) -> dict:
    """
    Tests state abstraction sensitivity to threshold variation.
    Uses an isolated random state
    Evaluates using balanced accuracy (mean of TPR and TNR) to avoid majority-class collapse in raw accuracy reporting.
    Returns dict of results for plotting.
    """
    py_state = random.getstate()
    np_state = np.random.get_state()
    random.seed(999); np.random.seed(999)

    parse_thresh_range = [0.40, 0.50, 0.60, 0.70, 0.75]
    perc_thresh_range  = [0.35, 0.40, 0.45, 0.50, 0.55]

    # Build labelled dataset: ground truth only from scenario fields
    dataset = []
    for _ in range(n_samples):
        diff  = random.choice(["easy", "mid", "hard"])
        sc    = generate_scenario(diff)
        noise = {"easy": 0.08, "mid": 0.15, "hard": 0.25}[diff]
        cmd   = simulate_command_parser(sc, noise)
        perc  = simulate_perception(sc, noise)

        required  = REQUIRED_PARAMS[sc["intent"]]
        n_req     = len(required)
        n_present = sum(1 for p in ["direction", "target"]
                        if p in required and sc.get(p) is not None)
        gt_clarify = (n_req > 0 and n_present < n_req) or \
                     sc.get("num_obstacles", 0) > 0 or \
                     (sc["intent"] == "NAVIGATE_TO" and sc.get("num_targets", 1) == 0)
        dataset.append((cmd, perc, gt_clarify))

    results_grid = {}
    for pt, pct in iterproduct(parse_thresh_range, perc_thresh_range):
        tp = tn = fp = fn = 0
        for cmd, perc, gt in dataset:
            # Inline state build with test thresholds
            required = REQUIRED_PARAMS[cmd.intent]
            n_pres   = len(cmd.present_params)
            n_req    = len(required)
            if not required:
              comp = Completeness.COMPLETE
            elif n_pres == n_req:
              comp = Completeness.COMPLETE
            elif n_pres == 0:
              comp = Completeness.INCOMPLETE
            else:
              comp = Completeness.PARTIAL

            if cmd.intent in UNAMBIGUOUS_INTENTS:
                clar = Clarity.CLEAR
            else:
                clar = Clarity.AMBIGUOUS if cmd.parse_confidence < pt else Clarity.CLEAR

            needs_tgt = cmd.intent == "NAVIGATE_TO"
            if not perc.path_clear:
                feas = Feasibility.BLOCKED
            elif needs_tgt:
                if not perc.target_visible:
                  feas = Feasibility.BLOCKED
                elif perc.num_target_matches > 1:
                  feas = Feasibility.UNCERTAIN
                elif perc.perception_confidence < pct + 0.05:
                  feas = Feasibility.UNCERTAIN
                else:
                  feas = Feasibility.FEASIBLE
            else:
                feas = Feasibility.UNCERTAIN if perc.perception_confidence < pct else Feasibility.FEASIBLE

            pred = oracle_should_clarify(comp, clar, feas)
            if pred and gt:
              tp += 1
            elif not pred and not gt:
              tn += 1
            elif pred and not gt:
              fp += 1
            else:
              fn += 1

        tpr = tp / (tp + fn + 1e-9)
        tnr = tn / (tn + fp + 1e-9)
        bal = (tpr + tnr) / 2
        raw = (tp + tn) / len(dataset)
        results_grid[(pt, pct)] = {"balanced_acc": bal, "raw_acc": raw,
                                    "tpr": tpr, "tnr": tnr}

    random.setstate(py_state)
    np.random.set_state(np_state)

    scores = [v["balanced_acc"] for v in results_grid.values()]
    print(f"\n  Sensitivity analysis: {len(results_grid)} threshold combinations")
    print(f"  Balanced accuracy range: {min(scores):.4f} – {max(scores):.4f}")
    print(f"  Score std: {np.std(scores):.4f}")
    print(f"  Conclusion: results are {'STABLE' if np.std(scores) < 0.02 else 'SENSITIVE'} "
          f"to threshold variation.")
    print(f"  Fixed thresholds (parse_ambiguous={PARSE_AMBIGUOUS_THRESHOLD}, "
          f"perc_uncertain={PERC_UNCERTAIN_THRESHOLD}) are justified by this stability.")

    return results_grid


def plot_sensitivity(results_grid: dict, path: str):
    """Heatmap of balanced accuracy across threshold combinations."""
    parse_vals = sorted(set(k[0] for k in results_grid))
    perc_vals  = sorted(set(k[1] for k in results_grid))

    matrix = np.array([[results_grid[(pt, pct)]["balanced_acc"]
                        for pct in perc_vals]
                       for pt in parse_vals])

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(matrix, cmap="YlOrRd", vmin=matrix.min() - 0.01,
                   vmax=matrix.max() + 0.01, aspect='auto')
    ax.set_xticks(range(len(perc_vals)))
    ax.set_xticklabels([f"{v:.2f}" for v in perc_vals], fontsize=9)
    ax.set_yticks(range(len(parse_vals)))
    ax.set_yticklabels([f"{v:.2f}" for v in parse_vals], fontsize=9)
    ax.set_xlabel("Perception Uncertainty Threshold", fontsize=10)
    ax.set_ylabel("Parse Ambiguity Threshold", fontsize=10)
    ax.set_title("Sensitivity Analysis: Balanced Accuracy\nAcross Threshold Combinations",
                 fontsize=12, fontweight='bold')

    for i in range(len(parse_vals)):
        for j in range(len(perc_vals)):
            ax.text(j, i, f"{matrix[i,j]:.3f}", ha='center', va='center',
                    fontsize=8, color='black')

    # Mark selected thresholds
    sel_i = parse_vals.index(PARSE_AMBIGUOUS_THRESHOLD) if PARSE_AMBIGUOUS_THRESHOLD in parse_vals else 2
    sel_j = perc_vals.index(PERC_UNCERTAIN_THRESHOLD)   if PERC_UNCERTAIN_THRESHOLD   in perc_vals else 2
    ax.add_patch(plt.Rectangle((sel_j - 0.5, sel_i - 0.5), 1, 1,
                                fill=False, edgecolor='blue', linewidth=2.5,
                                label=f"Selected ({PARSE_AMBIGUOUS_THRESHOLD}, {PERC_UNCERTAIN_THRESHOLD})"))
    ax.legend(loc='upper right', fontsize=9)
    plt.colorbar(im, ax=ax, label="Balanced accuracy")
    plt.tight_layout()
    plt.savefig(f"{path}/sensitivity_analysis.png", dpi=150)
    plt.close()

In [ ]:
# EXECUTION SUCCESS PROBABILITY MODEL
# P(success|state) = BASE × f_Completeness × f_Clarity × f_Feasibility
# Multiplicative factor model with each dimension contributing independently.

COMPLETENESS_FACTOR = {Completeness.COMPLETE: 1.00,
                       Completeness.PARTIAL:  0.72,
                       Completeness.INCOMPLETE: 0.38}
CLARITY_FACTOR      = {Clarity.CLEAR: 1.00, Clarity.AMBIGUOUS: 0.60}
FEASIBILITY_FACTOR  = {Feasibility.FEASIBLE: 1.00,
                       Feasibility.UNCERTAIN: 0.75,
                       Feasibility.BLOCKED:   0.15}
BASE_SUCCESS        = 0.95

def exec_success_prob(state: int) -> float:
    c, cl, f = decode_state(state)
    return float(np.clip(
        BASE_SUCCESS * COMPLETENESS_FACTOR[c] * CLARITY_FACTOR[cl] * FEASIBILITY_FACTOR[f],
        0.02, 0.99))

def print_success_prob_table():
    print("\n  Execution success probabilities (multiplicative factor model):")
    print(f"  {'ID':<4} {'Completeness':<14} {'Clarity':<11} {'Feasibility':<12} P(success)")
    print("  " + "─" * 56)
    for s in range(NUM_STATES):
        c, cl, f = decode_state(s)
        print(f"  {s:<4} {c.name:<14} {cl.name:<11} {f.name:<12} {exec_success_prob(s):.3f}")

In [ ]:
# Q-LEARNING DECISION MODULE
# Optimistic init: uniform [0, 0.1]; breaks action-0 tie bias.
# Linear epsilon decay over natural episodes only.
# Convergence tracking: max |ΔQ| per episode.

class QLearningAgent:
    """
    Tabular Q-learning agent

    Hyperparameters
      α = 0.1   (learning rate)
      γ = 0.9   (discount factor)
      ε = 0.2   (initial exploration)
    """
    def __init__(self, alpha: float = 0.1, gamma: float = 0.9,
                 epsilon_start: float = 0.20, epsilon_min: float = 0.05,
                 n_natural_episodes: int = 4445):
        self.alpha = alpha
        self.gamma  = gamma
        self.epsilon = epsilon_start
        self.epsilon_min   = epsilon_min
        self._decay_per_ep = (epsilon_start - epsilon_min) / (0.90 * n_natural_episodes)
        self.Q  = np.random.uniform(0.0, 0.1, (NUM_STATES, NUM_ACTIONS))
        self._prev_Q = self.Q.copy()
        self.q_delta_log = []
        # Policy stability tracking (correct convergence measure for stochastic RL)
        self._prev_policy = tuple(np.argmax(self.Q, axis=1))
        self._stable_count = 0

    def select_action(self, state: int, training: bool = True) -> int:
        """Epsilon greedy"""
        if training and random.random() < self.epsilon:
            return random.randint(0, 1)
        return int(np.argmax(self.Q[state]))

    def update_execute(self, state: int, reward: float):
        """
        Terminal Bellman update
        EXECUTE ends the episode.
        Q(s,a) = Q(s,a) + α[r − Q(s,a)]
        """
        self.Q[state, 0] += self.alpha * (reward - self.Q[state, 0])

    def update_clarify(self, state: int, next_state: int, reward: float):
        """
        Non-terminal Bellman update
        CLARIFY transitions to next state.
        Q(s,a) = Q(s,a) + α[r + γ·max Q(s',·) − Q(s,a)]
        """
        self.Q[state, 1] += self.alpha * (
            reward + self.gamma * np.max(self.Q[next_state]) - self.Q[state, 1]
        )

    def update_clarify_terminal(self, state: int, reward: float):
        """Terminal update for CLARIFY: updates Q[state, 1], not Q[state, 0]."""
        self.Q[state, 1] += self.alpha * (reward - self.Q[state, 1])

    def decay_epsilon(self):
        """Called only after natural episodes."""
        self.epsilon = max(self.epsilon - self._decay_per_ep, self.epsilon_min)

    def record_convergence(self):
        """
        Track policy stability after each natural episode.
        Policy stability measures onvergence in tabular Q-learning with stochastic rewards α>0 (argmax unchanged).

        Argmax of Q stabilises well before Q-values do (greedy policy).
        """
        current_policy = tuple(np.argmax(self.Q, axis=1))
        if current_policy == self._prev_policy:
            self._stable_count += 1
        else:
            self._stable_count = 0
            self._prev_policy  = current_policy
        # Log the max Q-change for the learning curve plot
        delta = float(np.max(np.abs(self.Q - self._prev_Q)))
        self.q_delta_log.append(delta)
        self._prev_Q = self.Q.copy()

    def is_converged(self, window: int = 500) -> bool:
        """
        Policy stability convergence: greedy policy (argmax Q) unchanged for 'window' consecutive natural episodes.
        500 natural episodes ≈ 5000 total × (8/9 natural rate) × last 11% of training.
        """
        return self._stable_count >= window

In [ ]:
# REWARD FUNCTION
# Execution: stochastic outcome from multiplicative probability model.
# Clarification: outcome-based. Reward depends on observed state improvement.
# Excessive clarification: episode failure, terminal update on Q[state, CLARIFY].

REWARD_EXEC_SUCCESS      = +10.0
REWARD_EXEC_FAIL         = -10.0
REWARD_CLAR_PRODUCTIVE   =  +3.0
REWARD_CLAR_UNPRODUCTIVE =  -2.0
REWARD_STEP_COST         =  -1.0
REWARD_EXCESSIVE_CLAR    =  -5.0

def compute_reward_execute(state: int, exec_scale: float) -> Tuple[float, bool]:
    prob    = min(exec_success_prob(state) * exec_scale, 0.99)
    success = random.random() < prob
    return (REWARD_EXEC_SUCCESS if success else REWARD_EXEC_FAIL), success

def compute_reward_clarify(state_before: int, state_after: int) -> float:
    """
    Outcome-based clarification reward. No oracle involvement.
    Productive iff the state uncertainty level genuinely decreased.
    """
    if state_uncertainty_level(state_after) < state_uncertainty_level(state_before):
        return REWARD_STEP_COST + REWARD_CLAR_PRODUCTIVE    # net +2
    else:
        return REWARD_STEP_COST + REWARD_CLAR_UNPRODUCTIVE  # net -3

In [ ]:
# SCENARIO GENERATOR
# Generates different locomotion scenarios covering all 18 states.

def generate_scenario(difficulty: str) -> dict:
    """
    Generates a hidden ground-truth locomotion scenario.
    easy: clear commands, uncluttered environment
    mid:  moderate ambiguity, some obstacles
    hard: frequent missing params, cluttered scenes

    The scenario dict is hidden from the Q-agent. Used only by simulate_command_parser() and simulate_perception().
    """
    cfg = {
        "easy": {"drop_prob": 0.10, "obs_pool": [0, 0, 0, 1], "tgt_pool": [1, 1, 1, 2]},
        "mid":  {"drop_prob": 0.25, "obs_pool": [0, 0, 1, 1, 2], "tgt_pool": [0, 1, 1, 2]},
        "hard": {"drop_prob": 0.42, "obs_pool": [0, 1, 1, 2, 2], "tgt_pool": [0, 0, 1, 2, 3]},
    }[difficulty]

    intent = random.choice(INTENTS)
    needed = REQUIRED_PARAMS[intent]

    direction = (random.choice(DIRECTIONS)
                 if "direction" in needed and random.random() > cfg["drop_prob"]
                 else None)
    target    = (random.choice(TARGETS)
                 if "target" in needed and random.random() > cfg["drop_prob"]
                 else None)
    distance  = (round(random.uniform(0.5, 3.0), 1)
                 if "direction" in needed and random.random() > 0.5
                 else None)

    return {
        "intent":        intent,
        "direction":     direction,
        "target":        target,
        "distance":      distance,
        "num_obstacles": random.choice(cfg["obs_pool"]),
        "num_targets":   random.choice(cfg["tgt_pool"]) if "target" in needed else 0,
    }

def generate_injected_scenario(state_id: int) -> dict:
    """
    Ensures training injection so that all 18 states receive Q-updates.
    Scenario that maps approximately to state_id for training coverage.
    """
    c, cl, f = decode_state(state_id)

    # Choose an intent that has parameters
    intent = random.choice(["MOVE", "TURN", "NAVIGATE_TO"])
    needed = REQUIRED_PARAMS[intent]

    # Completeness determines whether params are present
    if c == Completeness.COMPLETE:
        direction = random.choice(DIRECTIONS) if "direction" in needed else None
        target    = random.choice(TARGETS) if "target" in needed else None
    elif c == Completeness.PARTIAL and len(needed) > 1:
        direction = random.choice(DIRECTIONS) if "direction" in needed else None
        target    = None
    else:
        # INCOMPLETE
        direction = None
        target    = None

    # Feasibility determines obstacle and target counts
    num_obs = (0 if f == Feasibility.FEASIBLE else
               random.randint(1, 2) if f == Feasibility.BLOCKED else 0)
    num_tgt = (1 if f == Feasibility.FEASIBLE else
               0 if f == Feasibility.BLOCKED else
                random.choice([0, 2, 3]))  # UNCERTAIN: multiple or none

    return {
        "intent": intent, "direction": direction, "target": target,
        "distance": None, "num_obstacles": num_obs, "num_targets": num_tgt,
        "_inject_clarity": cl, #hint for parser to set confidence accordingly
    }

def simulate_command_parser_injected(scenario: dict, noise: float) -> LocoCommand:
    """
    Variant of simulate_command_parser.
    Overrides parse confidence to steer toward injected clarity dimension.
    """
    cmd = simulate_command_parser(scenario, noise)

    # Override confidence to hit the target Completeness and Clarity
    cl_hint = scenario.get("_inject_clarity")

    if cmd.intent not in UNAMBIGUOUS_INTENTS and cl_hint is not None:
        if cl_hint == Clarity.AMBIGUOUS:
            conf = float(np.clip(random.uniform(0.15, PARSE_AMBIGUOUS_THRESHOLD - 0.05), 0.05, 0.99))
        else:
            conf = float(np.clip(random.uniform(PARSE_AMBIGUOUS_THRESHOLD + 0.05, 0.95), 0.05, 0.99))
        cmd = LocoCommand(cmd.intent, cmd.direction, cmd.target, cmd.distance,
                          cmd.raw_text, conf, cmd.present_params, cmd.missing_params)
    return cmd

In [ ]:
# INTERACTION & EXECUTION LOOP
# Episode runner that implements the full perception-reasoning-action loop.

def run_episode(agent: QLearningAgent, difficulty: str,
                exec_scale: float, clarify_success: float,
                noise: float, training: bool = True,
                inject_state: Optional[int] = None) -> dict:
    """
    One complete SHRI episode.

    The loop:
      1. Receive command (STT)
      2. Parse command (NLU) -> LocoCommand
      3. Read sensors -> PerceptionState
      4. Fuse into state -> discrete state
      5. Q-agent decides: EXECUTE or CLARIFY
      6. If CLARIFY: apply reward, update scenario, observe next state, loop
      7. If EXECUTE: apply reward, update Q-table, end episode

    inject_state: forces the first state for training coverage.
    MAX_CLAR: limits clarification to few cycles.

    State built from LLM + sensor outputs every step.
    Clarification reward is outcome-based.
    Noise reduction after clarification only when resolution succeeded.
    Successful clarification sets num_obstacles=0.
    Excessive clarification terminates with penalty on Q[state, CLARIFY=1].
    Convergence recorded every episode during training.
    Epsilon decayed by caller on natural episodes only.
    """
    MAX_CLAR = 3

    scenario = (generate_injected_scenario(inject_state)
                if inject_state is not None else generate_scenario(difficulty))

    total_reward = 0.0
    n_clarify    = 0
    n_productive = 0
    exec_success = False
    did_execute  = False

    for step in range(MAX_CLAR + 1):
        # Perception & Language layer
        cmd  = (simulate_command_parser_injected(scenario, noise)
                if inject_state is not None and step == 0
                else simulate_command_parser(scenario, noise))
        perc = simulate_perception(scenario, noise)

        # State abstraction layer
        state = (inject_state if inject_state is not None and step == 0
                 else build_state(cmd, perc))

        # Q-learning decision layer
        action = agent.select_action(state, training=training)

        if action == 0:  # EXECUTE
            reward, success = compute_reward_execute(state, exec_scale)
            if training: agent.update_execute(state, reward)
            total_reward += reward
            exec_success  = success
            did_execute   = True
            break

        else:  # CLARIFY
            if step == MAX_CLAR:
                reward = REWARD_EXCESSIVE_CLAR
                if training: agent.update_clarify_terminal(state, reward)
                total_reward += reward
                break

            n_clarify += 1
            # Human's answer resolves the ambiguity
            resolution_succeeded = random.random() < clarify_success

            # Update scenario as human gave the missing information
            if resolution_succeeded:
                scenario["direction"]     = scenario.get("direction") or random.choice(DIRECTIONS)
                scenario["target"]        = scenario.get("target")    or random.choice(TARGETS)
                scenario["num_obstacles"] = 0   # full resolution
                scenario["num_targets"]   = 1

            next_noise = noise * 0.55 if resolution_succeeded else noise
            cmd_next   = simulate_command_parser(scenario, next_noise)
            perc_next  = simulate_perception(scenario, next_noise)
            next_s     = build_state(cmd_next, perc_next)

            reward = compute_reward_clarify(state, next_s)
            if state_uncertainty_level(next_s) < state_uncertainty_level(state):
                n_productive += 1

            if training: agent.update_clarify(state, next_s, reward)
            total_reward += reward

    return {
        "total_reward": total_reward,
        "exec_success": exec_success,
        "did_execute":  did_execute,
        "n_clarify":    n_clarify,
        "n_productive": n_productive,
        "clar_precision": (n_productive / n_clarify) if n_clarify > 0 else None,
    }

In [ ]:
# TRAINING AND EVALUATION LOOPS
# Evaluation uses 1000 episodes for stable metric estimation.

def train_agent(difficulty: str, exec_scale: float, clarify_success: float,
                noise: float, episodes: int = 5000,
                inject_every: int = 9) -> tuple:
    """
    Phase 1: Simulation-based training.
    inject_every: every Nth episode forces a random state for coverage.
    """
    n_natural = int(episodes * (1.0 - 1.0 / inject_every))
    agent     = QLearningAgent(n_natural_episodes=n_natural)

    natural_rewards = []

    for ep in range(episodes):
        is_inj = (ep % inject_every == 0)
        inject = random.choice(ALL_STATES) if is_inj else None

        result = run_episode(agent, difficulty, exec_scale, clarify_success,
                             noise, training=True, inject_state=inject)

        if not is_inj:
            natural_rewards.append(result["total_reward"])
            agent.decay_epsilon()  # decay only on natural episodes
            agent.record_convergence()  # convergence tracked on natural episodes only

    return agent, np.array(natural_rewards), agent.is_converged()

def evaluate_agent(agent: QLearningAgent, difficulty: str,
                   exec_scale: float, clarify_success: float,
                   noise: float, episodes: int = 1000) -> dict:
    #Phase 2: Greedy evaluation with Q-table.
    exec_ok = exec_fail = clar_tot = clar_prod = excess = 0
    rewards = []

    for _ in range(episodes):
        result = run_episode(agent, difficulty, exec_scale, clarify_success,
                             noise, training=False)
        if result["did_execute"]:
            if result["exec_success"]:
              exec_ok   += 1
            else:
              exec_fail += 1
        else:
            excess += 1
        clar_tot  += result["n_clarify"]
        clar_prod += result["n_productive"]
        rewards.append(result["total_reward"])

    te = exec_ok + exec_fail + 1e-9
    tc = clar_tot

    return {
        "task_success_rate":            round(exec_ok  / te, 4),
        "failure_rate":                 round(exec_fail / te, 4),
        "clarification_rate":           round(clar_tot / episodes, 3),
        "clarification_precision":      round(clar_prod / tc, 4) if tc > 0 else None,
        "excessive_clarification_rate": round(excess / episodes, 4),
        "mean_episode_reward":          round(float(np.mean(rewards)), 3),
    }

def run_experiment(difficulty: str, exec_scale: float, clarify_success: float,
                   noise: float, runs: int = 10, episodes: int = 5000) -> dict:
    #Multi-seed experiment. Averages metrics.
    all_agents, all_metrics, all_rewards, n_conv = [], [], [], 0

    for seed_i in range(runs):
        random.seed(SEED + seed_i)
        np.random.seed(SEED + seed_i)

        agent, nat_rewards, converged = train_agent(
            difficulty, exec_scale, clarify_success, noise, episodes)
        metrics = evaluate_agent(agent, difficulty, exec_scale,
                                 clarify_success, noise, episodes=1000)
        all_agents.append(agent)
        all_metrics.append(metrics)
        all_rewards.append(nat_rewards)
        if converged: n_conv += 1

    random.seed(SEED); np.random.seed(SEED)

    max_len     = max(len(r) for r in all_rewards)
    padded      = [np.pad(r, (0, max_len - len(r)), constant_values=np.nan) for r in all_rewards]
    avg_rewards = np.nanmean(padded, axis=0)

    numeric_keys = [k for k in all_metrics[0] if isinstance(all_metrics[0][k], float)]
    avg = {k: round(float(np.mean([m[k] for m in all_metrics if m[k] is not None])), 4)
           for k in numeric_keys}
    std = {k: round(float(np.std( [m[k] for m in all_metrics if m[k] is not None])), 4)
           for k in numeric_keys}

    ranked       = sorted(zip([m["task_success_rate"] for m in all_metrics], all_agents),
                          key=lambda x: x[0])
    median_agent = ranked[len(ranked) // 2][1]

    print(f"  Convergence: {n_conv}/{runs} seeds converged within {episodes} episodes")

    return {"agent": median_agent, "avg_rewards": avg_rewards,
            "avg_metrics": avg, "std_metrics": std, "n_converged": n_conv}

In [ ]:
#POLICY REPORTING
def report_policy(agent: QLearningAgent, env_name: str, path: str) -> float:
    """Reports all 18 states with oracle correctness annotation."""
    lines = [
        f"\nLearned Policy: {env_name.upper()} environment",
        "=" * 88,
        f"  {'✓/✗':<4} {'Completeness':<14} {'Clarity':<11} {'Feasibility':<12}"
        f"{'Decision':<10} {'Q(EXEC)':>9} {'Q(CLAR)':>9}  Oracle",
        "  " + "─" * 84,
    ]
    correct = 0
    for s in range(NUM_STATES):
        c, cl, f  = decode_state(s)
        action    = int(np.argmax(agent.Q[s]))
        act_str   = "EXECUTE" if action == 0 else "CLARIFY"
        oracle    = "CLARIFY" if oracle_should_clarify(c, cl, f) else "EXECUTE"
        match     = "✓" if act_str == oracle else "✗"
        flag      = "  ← review" if act_str != oracle else ""
        lines.append(
            f"  {match:<4} {c.name:<14} {cl.name:<11} {f.name:<12}"
            f"{act_str:<10} {agent.Q[s,0]:>+9.3f} {agent.Q[s,1]:>+9.3f}  ({oracle}){flag}")
        if act_str == oracle: correct += 1

    acc = correct / NUM_STATES
    lines += ["",
              f"  Policy accuracy: {correct}/{NUM_STATES} ({100*acc:.0f}%)",
              f"  EXECUTE: COMPLETE + CLEAR + FEASIBLE only (state 0: single optimal execution state)",
              f"  CLARIFY: all other 17 states (INCOMPLETE, PARTIAL, AMBIGUOUS, or BLOCKED)"]
    text = "\n".join(lines)
    print(text)
    with open(f"{path}/{env_name}_policy.txt", "w", encoding="utf-8") as f:
        f.write(text)
    return acc

In [ ]:
# VISUALISATIONS
def plot_learning_curve(rewards: np.ndarray, q_deltas: list,
                        env_name: str, path: str):
    def ma(x, w=100):
        return np.convolve(x, np.ones(w)/w, mode='valid')

    valid = rewards[~np.isnan(rewards)]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=False)

    ax1.plot(valid, alpha=0.06, color='#2563eb', linewidth=0.5)
    if len(valid) >= 100:
        ax1.plot(ma(valid), color='#2563eb', linewidth=2.2, label="Smoothed (w=100)")
    ax1.axhline(0, color='#6b7280', linewidth=0.7, linestyle='--', alpha=0.5)
    ax1.set_title(f"Learning Curve: {env_name.capitalize()} (natural episodes only)",
                  fontsize=12)
    ax1.set_ylabel("Episode Reward"); ax1.legend(); ax1.grid(alpha=0.18)

    ax2.semilogy(q_deltas, color='#7c3aed', linewidth=1.2, alpha=0.7)
    ax2.axhline(0.005, color='#dc2626', linestyle='--', linewidth=1.2,
                label="Convergence threshold (0.005)")
    ax2.set_title("Q-Value Max Change per Episode (Convergence Monitor)", fontsize=11)
    ax2.set_xlabel("Episode"); ax2.set_ylabel("Max |ΔQ| (log scale)")
    ax2.legend(); ax2.grid(alpha=0.18)

    plt.tight_layout()
    plt.savefig(f"{path}/{env_name}_learning_curve.png", dpi=150)
    plt.close()


def plot_policy_heatmap(agent: QLearningAgent, env_name: str, path: str):
    policy = np.argmax(agent.Q, axis=1).reshape(3, 2, 3)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    titles = ["Completeness: COMPLETE", "Completeness: PARTIAL", "Completeness: INCOMPLETE"]
    for i, (ax, title) in enumerate(zip(axes, titles)):
        ax.imshow(policy[i], cmap="RdBu_r", vmin=0, vmax=1, aspect='auto')
        ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
        ax.set_xticks([0,1,2]); ax.set_xticklabels(["FEASIBLE","UNCERTAIN","BLOCKED"], fontsize=8)
        ax.set_yticks([0,1]);   ax.set_yticklabels(["CLEAR","AMBIGUOUS"], fontsize=9)
        ax.set_xlabel("Environment Feasibility", fontsize=9)
        if i == 0: ax.set_ylabel("Intent Clarity", fontsize=9)
        for r in range(2):
            for c_idx in range(3):
                lbl   = "EXEC" if policy[i,r,c_idx]==0 else "CLAR"
                color = '#111827' if policy[i,r,c_idx]==0 else '#f9fafb'
                ax.text(c_idx, r, lbl, ha='center', va='center',
                        fontsize=9, fontweight='bold', color=color)
    ep = mpatches.Patch(color='#3b82f6', label='EXECUTE')
    cp = mpatches.Patch(color='#dc2626', label='CLARIFY')
    fig.legend(handles=[ep,cp], loc='lower center', ncol=2,
               fontsize=10, bbox_to_anchor=(0.5,-0.01))
    plt.suptitle(f"Learned Q-Policy: {env_name.capitalize()} (All 18 States)",
                 fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0,0.06,1,0.97])
    plt.savefig(f"{path}/{env_name}_policy_heatmap.png", dpi=150, bbox_inches='tight')
    plt.close()


def plot_q_values(agent: QLearningAgent, env_name: str, path: str):
    #Bar chart of Q(EXECUTE) vs Q(CLARIFY) for all active states.
    labels = [f"{decode_state(s)[0].name[:4]}\n{decode_state(s)[1].name[:4]}\n{decode_state(s)[2].name[:4]}"
              for s in range(NUM_STATES)]
    qe = [agent.Q[s,0] for s in range(NUM_STATES)]
    qc = [agent.Q[s,1] for s in range(NUM_STATES)]
    x, w = np.arange(NUM_STATES), 0.35
    fig, ax = plt.subplots(figsize=(18,5))
    ax.bar(x-w/2, qe, w, label='Q(EXECUTE)', color='#3b82f6', alpha=0.85)
    ax.bar(x+w/2, qc, w, label='Q(CLARIFY)', color='#dc2626', alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7)
    ax.set_title(f"Learned Q-Values: {env_name.capitalize()} (All 18 States)",
                 fontsize=12, fontweight='bold')
    ax.set_ylabel("Q-Value"); ax.legend(); ax.grid(axis='y', alpha=0.18)
    plt.tight_layout()
    plt.savefig(f"{path}/{env_name}_q_values.png", dpi=150)
    plt.close()


def plot_combined_results(results: dict, path: str):
    colors = {"easy":"#16a34a","mid":"#d97706","hard":"#dc2626"}
    def ma(x, w=80): return np.convolve(x, np.ones(w)/w, mode='valid')

    fig, axes = plt.subplots(1, 2, figsize=(15,5))
    for name, data in results.items():
        valid = data["avg_rewards"][~np.isnan(data["avg_rewards"])]
        if len(valid) >= 80:
            axes[0].plot(ma(valid), label=name.capitalize(),
                         color=colors[name], linewidth=2.2)
    axes[0].axhline(0, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
    axes[0].set_title("Combined Learning Curves (Smoothed, w=80)\nNatural episodes only",
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Natural episode"); axes[0].set_ylabel("Avg Episode Reward")
    axes[0].legend(fontsize=11); axes[0].grid(alpha=0.18)

    metric_keys  = ["task_success_rate","clarification_rate","clarification_precision"]
    metric_names = ["Task\nSuccess Rate","Clarif.\nRate/ep","Clarif.\nPrecision"]
    x, w = np.arange(3), 0.25
    for i,(name,data) in enumerate(results.items()):
        vals = [data["avg_metrics"].get(k,0) for k in metric_keys]
        errs = [data["std_metrics"].get(k,0)  for k in metric_keys]
        axes[1].bar(x+i*w, vals, w, yerr=errs, label=name.capitalize(),
                    color=colors[name], alpha=0.85, capsize=4, ecolor='#374151')
    axes[1].set_xticks(x+w); axes[1].set_xticklabels(metric_names, fontsize=9)
    axes[1].set_ylim(0,1.28)
    axes[1].set_title("Key Metrics (Mean ± Std, 10 Seeds)", fontsize=12, fontweight='bold')
    axes[1].set_ylabel("Score"); axes[1].legend(fontsize=11); axes[1].grid(axis='y',alpha=0.18)
    plt.tight_layout()
    plt.savefig(f"{path}/combined_results.png", dpi=200, bbox_inches='tight')
    plt.close()

In [ ]:
# MAIN
if __name__ == "__main__":
    BASE_PATH  = "/PATH"
    RUN_ID       = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    RESULTS_PATH = os.path.join(BASE_PATH, f"run_{RUN_ID}")
    os.makedirs(RESULTS_PATH, exist_ok=True)

    random.seed(SEED); np.random.seed(SEED)

    print("=" * 65)
    print("  SHRI Q-Learning Execute-or-Clarify Decision Module")
    print("  Robot: Unitree Go2 (Quadruped) Locomotive Task Domain")
    print(f"  Results → {RESULTS_PATH}")
    print("=" * 65)

    # Step 1: Sensitivity analysis (isolated random state)
    print("\n  Step 1: Threshold sensitivity analysis...")
    sens_results = run_sensitivity_analysis(n_samples=3000)
    plot_sensitivity(sens_results, RESULTS_PATH)

    # Step 2: Success probability table
    print_success_prob_table()

    # Step 3: Full state space reference table
    print(f"\n  {'ID':>3}  {'Completeness':<14} {'Clarity':<12} {'Feasibility':<12}  P(succ)  Oracle")
    print(f"  Note: EXECUTE is optimal only for COMPLETE+CLEAR+FEASIBLE (state 0).")
    print(f"        All other states benefit from clarification given the reward structure.")
    print("  " + "─" * 68)
    for s in range(NUM_STATES):
        c, cl, f = decode_state(s)
        oracle   = "CLARIFY" if oracle_should_clarify(c,cl,f) else "EXECUTE"
        print(f"  {s:>3}  {c.name:<14} {cl.name:<12} {f.name:<12}  "
              f"{exec_success_prob(s):.3f}    {oracle}")

    # Step 4: Experiments
    environments = {
        "easy": {"exec_scale":1.10,"clarify_success":0.90,"noise":0.08,
                 "label":"Easy: clear commands, uncluttered environment"},
        "mid":  {"exec_scale":1.00,"clarify_success":0.75,"noise":0.15,
                 "label":"Mid: moderate ambiguity, some obstacles"},
        "hard": {"exec_scale":0.85,"clarify_success":0.60,"noise":0.25,
                 "label":"Hard: frequent missing params, cluttered scene"},
    }

    results     = {}
    all_summary = []

    for env_name, cfg in environments.items():
        print(f"\n{'─'*65}\n  {cfg['label']}\n{'─'*65}")

        data = run_experiment(
            difficulty=env_name, exec_scale=cfg["exec_scale"],
            clarify_success=cfg["clarify_success"], noise=cfg["noise"],
            runs=10, episodes=5000)

        print(f"\n  Results (mean ± std, 10 seeds):")
        for k, v in data["avg_metrics"].items():
            s = data["std_metrics"].get(k, "N/A")
            print(f"    {k:<40}: {v:.4f} ± {s:.4f}" if isinstance(v,float)
                  else f"    {k:<40}: {v}")

        acc = report_policy(data["agent"], env_name, RESULTS_PATH)
        plot_learning_curve(data["avg_rewards"], data["agent"].q_delta_log,
                            env_name, RESULTS_PATH)
        plot_policy_heatmap(data["agent"], env_name, RESULTS_PATH)
        plot_q_values(data["agent"], env_name, RESULTS_PATH)

        results[env_name] = data
        all_summary.append({
            "environment":     env_name,
            "policy_accuracy": round(acc, 4),
            "n_converged":     data["n_converged"],
            **{k:v for k,v in data["avg_metrics"].items() if isinstance(v,float)},
        })

    plot_combined_results(results, RESULTS_PATH)

    # Save Q-tables for ROS2 loading
    for env_name, data in results.items():
        np.save(f"{RESULTS_PATH}/{env_name}_q_table.npy", data["agent"].Q)
        print(f"  Saved: {env_name}_q_table.npy")

     # Save metrics CSV
    with open(f"{RESULTS_PATH}/metrics_summary.csv","w",
              newline="",encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=all_summary[0].keys())
        writer.writeheader()
        writer.writerows(all_summary)

    print(f"\n{'='*65}")
    print("  Complete. Q-tables ready for ROS2 deployment.")
    print(f"{'='*65}")